In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import pandas as pd

project_path = "/content/drive/MyDrive/turkish_legal_rag"
processed_path = f"{project_path}/data/processed"

print(os.listdir(processed_path))

['val_qa.csv', 'test_qa.csv', 'train_qa.csv', 'kaggle_train_qa.csv', 'kaggle_val_qa.csv', 'kaggle_test_qa.csv', 'retrieval_corpus.csv']


In [3]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.4 MB/s eta 0:00:00


In [4]:
chunks_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")

print(chunks_df.shape)
chunks_df.head()

(3775, 5)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


In [5]:
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)

print("Model loaded:", embedding_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [6]:
chunk_texts = chunks_df["chunk_text"].astype(str).tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Embeddings shape: (3775, 384)


In [7]:
import faiss
import numpy as np

chunk_embeddings = chunk_embeddings.astype("float32")

# Cosine similarity için normalize + Inner Product index
faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("FAISS index total:", index.ntotal)

FAISS index total: 3775


In [8]:
os.makedirs(f"{project_path}/outputs/faiss", exist_ok=True)

faiss.write_index(index, f"{project_path}/outputs/faiss/baseline_faiss.index")
np.save(f"{project_path}/outputs/faiss/baseline_chunk_embeddings.npy", chunk_embeddings)

print("FAISS index and embeddings saved.")

FAISS index and embeddings saved.


In [9]:
def retrieve_top_k(query, model, index, chunks_df, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(score),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [10]:
test_queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanı kanunları kaç gün içinde yayımlar?",
    "Türkiye Büyük Millet Meclisinin görevleri nelerdir?"
]

for q in test_queries:
    print("=" * 120)
    print("QUERY:", q)
    results = retrieve_top_k(q, embedding_model, index, chunks_df, k=3)

    for item in results:
        print(f"Rank {item['rank']} | Score: {item['score']:.4f} | {item['chunk_id']}")
        print(item["chunk_text"][:500])
        print("-" * 80)

QUERY: Egemenlik kime aittir?
Rank 1 | Score: 0.7146 | chunk_000002
Millet iradesinin mutlak üstünlüğü, egemenliğin kayıtsız şartsız Türk Milletine ait olduğu ve bunu millet adına kullanmaya yetkili kılınan hiçbir kişi ve kuruluşun, bu Anayasada gösterilen hürriyetçi demokrasi ve bunun icaplarıyla belirlenmiş hukuk düzeni dışına çıkamayacağı;
--------------------------------------------------------------------------------
Rank 2 | Score: 0.6703 | chunk_000399
İKİNCİ KISIM TEMEL HAKLAR VE ÖDEVLER
İKİNCİ BÖLÜM Kişinin Hakları ve Ödevleri
Öncesi… VI I. Düşünce ve kanaat hürriyeti
--------------------------------------------------------------------------------
Rank 3 | Score: 0.6519 | chunk_000269
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
QUERY: Türkiye Cumhuriyetinin yönetim şekli nedir?
Rank 1 | Score: 0

In [11]:
manual_eval_df = pd.DataFrame([
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir.",
        "gold_keywords": ["egemenlik", "kayıtsız", "şartsız", "milletindir"]
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir.",
        "gold_keywords": ["türkiye", "devleti", "cumhuriyettir"]
    },
    {
        "question": "Cumhurbaşkanı kanunları kaç gün içinde yayımlar?",
        "expected_answer": "Cumhurbaşkanı, Türkiye Büyük Millet Meclisince kabul edilen kanunları onbeş gün içinde yayımlar.",
        "gold_keywords": ["cumhurbaşkanı", "kanunları", "onbeş", "gün", "yayımlar"]
    },
    {
        "question": "Türkiye Büyük Millet Meclisinin görevleri nelerdir?",
        "expected_answer": "Türkiye Büyük Millet Meclisinin görev ve yetkileri kanun koymak, değiştirmek ve kaldırmak; bütçe ve kesinhesap kanun tekliflerini görüşmek ve kabul etmek; para basılmasına ve savaş ilanına karar vermek gibi görevleri içerir.",
        "gold_keywords": ["kanun", "koymak", "değiştirmek", "kaldırmak", "bütçe", "savaş", "andlaşmalar"]
    }
])

manual_eval_df

,question,expected_answer,gold_keywords
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,"[egemenlik, kayıtsız, şartsız, milletindir]"
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,"[türkiye, devleti, cumhuriyettir]"
2,Cumhurbaşkanı kanunları kaç gün içinde yayımlar?,"Cumhurbaşkanı, Türkiye Büyük Millet Meclisince...","[cumhurbaşkanı, kanunları, onbeş, gün, yayımlar]"
3,Türkiye Büyük Millet Meclisinin görevleri nele...,Türkiye Büyük Millet Meclisinin görev ve yetki...,"[kanun, koymak, değiştirmek, kaldırmak, bütçe,..."


In [12]:
def keyword_hit_at_k(question, gold_keywords, k=5):
    results = retrieve_top_k(question, embedding_model, index, chunks_df, k=k)
    retrieved_text = " ".join([r["chunk_text"].lower() for r in results])

    hits = 0
    for kw in gold_keywords:
        if kw.lower() in retrieved_text:
            hits += 1

    return hits / len(gold_keywords), results


eval_rows = []

for _, row in manual_eval_df.iterrows():
    score_at_1, results_1 = keyword_hit_at_k(row["question"], row["gold_keywords"], k=1)
    score_at_3, results_3 = keyword_hit_at_k(row["question"], row["gold_keywords"], k=3)
    score_at_5, results_5 = keyword_hit_at_k(row["question"], row["gold_keywords"], k=5)

    eval_rows.append({
        "question": row["question"],
        "keyword_score_at_1": score_at_1,
        "keyword_score_at_3": score_at_3,
        "keyword_score_at_5": score_at_5,
        "top1_chunk": results_1[0]["chunk_id"],
        "top1_text": results_1[0]["chunk_text"]
    })

retrieval_eval_df = pd.DataFrame(eval_rows)
retrieval_eval_df

,question,keyword_score_at_1,keyword_score_at_3,keyword_score_at_5,top1_chunk,top1_text
0,Egemenlik kime aittir?,0.5,1.0,1.0,chunk_000002,"Millet iradesinin mutlak üstünlüğü, egemenliği..."
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,1.0,1.0,1.0,chunk_000264,Madde 1 – Türkiye Devleti bir Cumhuriyettir. I...
2,Cumhurbaşkanı kanunları kaç gün içinde yayımlar?,0.0,0.2,0.4,chunk_000618,Madde 31- Bu Kanunun uygulanması ile ilgili es...
3,Türkiye Büyük Millet Meclisinin görevleri nele...,0.0,1.0,1.0,chunk_000031,"Türkiye Büyük Millet Meclisinin bütün bina, te..."


In [13]:
baseline_metrics = {
    "Keyword Accuracy@1": retrieval_eval_df["keyword_score_at_1"].mean(),
    "Keyword Accuracy@3": retrieval_eval_df["keyword_score_at_3"].mean(),
    "Keyword Accuracy@5": retrieval_eval_df["keyword_score_at_5"].mean(),
}

baseline_metrics

{'Keyword Accuracy@1': np.float64(0.375),
 'Keyword Accuracy@3': np.float64(0.8),
 'Keyword Accuracy@5': np.float64(0.85)}

In [14]:
metrics_df = pd.DataFrame([
    {"metric": key, "value": round(value, 4)}
    for key, value in baseline_metrics.items()
])

metrics_df.to_csv(
    f"{project_path}/outputs/metrics/baseline_retrieval_keyword_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

retrieval_eval_df.to_csv(
    f"{project_path}/outputs/metrics/baseline_retrieval_details.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics_df

,metric,value
0,Keyword Accuracy@1,0.375
1,Keyword Accuracy@3,0.800
2,Keyword Accuracy@5,0.850


In [15]:
!pip install rank-bm25

In [16]:
from rank_bm25 import BM25Okapi
import re
import numpy as np
import pandas as pd

def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    tokens = [t for t in tokens if t not in stopwords and len(t) > 1]
    return tokens

In [17]:
tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df["chunk_text"].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(tokenized_corpus))

BM25 corpus size: 3775


In [18]:
def bm25_retrieve_top_k(query, chunks_df, bm25, k=5):
    tokenized_query = simple_turkish_tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(scores[idx]),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [19]:
test_queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanı kanunları kaç gün içinde yayımlar?",
    "Türkiye Büyük Millet Meclisinin görevleri nelerdir?"
]

for q in test_queries:
    print("=" * 120)
    print("QUERY:", q)

    results = bm25_retrieve_top_k(q, chunks_df, bm25, k=3)

    for item in results:
        print(f"Rank {item['rank']} | Score: {item['score']:.4f} | {item['chunk_id']}")
        print(item["chunk_text"][:500])
        print("-" * 80)

QUERY: Egemenlik kime aittir?
Rank 1 | Score: 8.4252 | chunk_000269
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
Rank 2 | Score: 8.3009 | chunk_003519
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | Score: 7.2579 | chunk_001360
Madde 12 – (1) Davaya bakmak yetkisi, suçun işlendiği yer mahkemesine aittir.
--------------------------------------------------------------------------------
QUERY: Türkiye Cumhuriyetinin yönetim şekli nedir?
Rank 1 | Score: 11.5179 | chunk_000001
Dünya milletleri ailesinin eşit haklara sahip şerefli bir üyesi olarak, Türkiye Cumhuriyetinin ebedi varlığı, refahı, maddi ve manevi mutluluğu ile

In [20]:
def bm25_keyword_hit_at_k(question, gold_keywords, k=5):
    results = bm25_retrieve_top_k(question, chunks_df, bm25, k=k)
    retrieved_text = " ".join([r["chunk_text"].lower() for r in results])

    hits = 0
    for kw in gold_keywords:
        if kw.lower() in retrieved_text:
            hits += 1

    return hits / len(gold_keywords), results


bm25_eval_rows = []

for _, row in manual_eval_df.iterrows():
    score_at_1, results_1 = bm25_keyword_hit_at_k(row["question"], row["gold_keywords"], k=1)
    score_at_3, results_3 = bm25_keyword_hit_at_k(row["question"], row["gold_keywords"], k=3)
    score_at_5, results_5 = bm25_keyword_hit_at_k(row["question"], row["gold_keywords"], k=5)

    bm25_eval_rows.append({
        "question": row["question"],
        "keyword_score_at_1": score_at_1,
        "keyword_score_at_3": score_at_3,
        "keyword_score_at_5": score_at_5,
        "top1_chunk": results_1[0]["chunk_id"],
        "top1_text": results_1[0]["chunk_text"]
    })

bm25_eval_df = pd.DataFrame(bm25_eval_rows)
bm25_eval_df

,question,keyword_score_at_1,keyword_score_at_3,keyword_score_at_5,top1_chunk,top1_text
0,Egemenlik kime aittir?,1.000000,1.000000,1.000000,chunk_000269,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin..."
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,0.333333,0.333333,0.666667,chunk_000001,Dünya milletleri ailesinin eşit haklara sahip ...
2,Cumhurbaşkanı kanunları kaç gün içinde yayımlar?,1.000000,1.000000,1.000000,chunk_000010,"Madde 89 – Cumhurbaşkanı, Türkiye Büyük Millet..."
3,Türkiye Büyük Millet Meclisinin görevleri nele...,0.142857,0.142857,1.000000,chunk_000294,Madde 9 – İlk genel seçimler sonucu toplanacak...


In [21]:
bm25_metrics = {
    "Keyword Accuracy@1": bm25_eval_df["keyword_score_at_1"].mean(),
    "Keyword Accuracy@3": bm25_eval_df["keyword_score_at_3"].mean(),
    "Keyword Accuracy@5": bm25_eval_df["keyword_score_at_5"].mean(),
}

bm25_metrics

{'Keyword Accuracy@1': np.float64(0.619047619047619),
 'Keyword Accuracy@3': np.float64(0.619047619047619),
 'Keyword Accuracy@5': np.float64(0.9166666666666666)}

In [22]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())

In [23]:
def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    # Dense scores
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    # BM25 scores
    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    # Normalize
    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    # Hybrid score
    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [24]:
test_queries = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanı kanunları kaç gün içinde yayımlar?",
    "Türkiye Büyük Millet Meclisinin görevleri nelerdir?"
]

for q in test_queries:
    print("=" * 120)
    print("QUERY:", q)

    results = hybrid_retrieve_top_k(
        q,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=3,
        alpha=0.5
    )

    for item in results:
        print(
            f"Rank {item['rank']} | Score: {item['score']:.4f} | "
            f"Dense: {item['dense_score']:.4f} | BM25: {item['bm25_score']:.4f} | "
            f"{item['chunk_id']}"
        )
        print(item["chunk_text"][:500])
        print("-" * 80)

QUERY: Egemenlik kime aittir?
Rank 1 | Score: 0.9584 | Dense: 0.9167 | BM25: 1.0000 | chunk_000269
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
Rank 2 | Score: 0.8981 | Dense: 0.8109 | BM25: 0.9852 | chunk_003519
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
Rank 3 | Score: 0.8044 | Dense: 0.8691 | BM25: 0.7397 | chunk_002321
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme
--------------------------------------------------------------------------------
QUERY: Türkiye Cumhuriyetinin yönetim şekli nedir?
Rank 1 | Score: 0.

In [25]:
def hybrid_keyword_hit_at_k(question, gold_keywords, k=5, alpha=0.5):
    results = hybrid_retrieve_top_k(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=k,
        alpha=alpha
    )

    retrieved_text = " ".join([r["chunk_text"].lower() for r in results])

    hits = 0
    for kw in gold_keywords:
        if kw.lower() in retrieved_text:
            hits += 1

    return hits / len(gold_keywords), results

In [26]:
hybrid_eval_rows = []

for _, row in manual_eval_df.iterrows():
    score_at_1, results_1 = hybrid_keyword_hit_at_k(row["question"], row["gold_keywords"], k=1, alpha=0.5)
    score_at_3, results_3 = hybrid_keyword_hit_at_k(row["question"], row["gold_keywords"], k=3, alpha=0.5)
    score_at_5, results_5 = hybrid_keyword_hit_at_k(row["question"], row["gold_keywords"], k=5, alpha=0.5)

    hybrid_eval_rows.append({
        "question": row["question"],
        "keyword_score_at_1": score_at_1,
        "keyword_score_at_3": score_at_3,
        "keyword_score_at_5": score_at_5,
        "top1_chunk": results_1[0]["chunk_id"],
        "top1_text": results_1[0]["chunk_text"]
    })

hybrid_eval_df = pd.DataFrame(hybrid_eval_rows)
hybrid_eval_df

,question,keyword_score_at_1,keyword_score_at_3,keyword_score_at_5,top1_chunk,top1_text
0,Egemenlik kime aittir?,1.000000,1.000000,1.000000,chunk_000269,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin..."
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,0.666667,0.666667,0.666667,chunk_000000,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Cumhurbaşkanı kanunları kaç gün içinde yayımlar?,1.000000,1.000000,1.000000,chunk_000010,"Madde 89 – Cumhurbaşkanı, Türkiye Büyük Millet..."
3,Türkiye Büyük Millet Meclisinin görevleri nele...,1.000000,1.000000,1.000000,chunk_000008,Türkiye Büyük Millet Meclisinin görev ve yetki...


In [27]:
hybrid_metrics = {
    "Keyword Accuracy@1": hybrid_eval_df["keyword_score_at_1"].mean(),
    "Keyword Accuracy@3": hybrid_eval_df["keyword_score_at_3"].mean(),
    "Keyword Accuracy@5": hybrid_eval_df["keyword_score_at_5"].mean(),
}

hybrid_metrics

{'Keyword Accuracy@1': np.float64(0.9166666666666666),
 'Keyword Accuracy@3': np.float64(0.9166666666666666),
 'Keyword Accuracy@5': np.float64(0.9166666666666666)}

In [28]:
hybrid_metrics_df = pd.DataFrame([
    {"method": "Hybrid Dense+BM25", "metric": key, "value": round(value, 4)}
    for key, value in hybrid_metrics.items()
])

hybrid_metrics_df.to_csv(
    f"{project_path}/outputs/metrics/hybrid_retrieval_keyword_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

hybrid_eval_df.to_csv(
    f"{project_path}/outputs/metrics/hybrid_retrieval_details.csv",
    index=False,
    encoding="utf-8-sig"
)

hybrid_metrics_df

,method,metric,value
0,Hybrid Dense+BM25,Keyword Accuracy@1,0.9167
1,Hybrid Dense+BM25,Keyword Accuracy@3,0.9167
2,Hybrid Dense+BM25,Keyword Accuracy@5,0.9167


In [29]:
comparison_df = pd.DataFrame([
    {
        "method": "Dense Retrieval",
        "accuracy_at_1": baseline_metrics["Keyword Accuracy@1"],
        "accuracy_at_3": baseline_metrics["Keyword Accuracy@3"],
        "accuracy_at_5": baseline_metrics["Keyword Accuracy@5"],
    },
    {
        "method": "BM25 Retrieval",
        "accuracy_at_1": bm25_metrics["Keyword Accuracy@1"],
        "accuracy_at_3": bm25_metrics["Keyword Accuracy@3"],
        "accuracy_at_5": bm25_metrics["Keyword Accuracy@5"],
    },
    {
        "method": "Hybrid Dense + BM25",
        "accuracy_at_1": hybrid_metrics["Keyword Accuracy@1"],
        "accuracy_at_3": hybrid_metrics["Keyword Accuracy@3"],
        "accuracy_at_5": hybrid_metrics["Keyword Accuracy@5"],
    }
])

comparison_df

,method,accuracy_at_1,accuracy_at_3,accuracy_at_5
0,Dense Retrieval,0.375000,0.800000,0.850000
1,BM25 Retrieval,0.619048,0.619048,0.916667
2,Hybrid Dense + BM25,0.916667,0.916667,0.916667


In [30]:
comparison_df.to_csv(
    f"{project_path}/outputs/metrics/retrieval_comparison_keyword_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Retrieval comparison saved.")

Retrieval comparison saved.
